# CourtVision -- entrainement YOLO11s-pose : les 14 points du terrain de tennis

**Avant de lancer :**
1. Sur Google Drive, cree le dossier **`CourtVision_court`** a la racine de *Mon Drive* et depose-y
   **`court_pose.zip`** (le fichier `C:\Code\CourtVision\court_pose.zip` de ton PC).
2. Ici : *Execution* > *Modifier le type d'execution* > **GPU T4** > Enregistrer.
3. *Execution* > **Tout executer**. Dans la fenetre Google : ton compte, puis **Tout selectionner** > Continuer.

- Tout est enregistre dans `Mon Drive/CourtVision_court/runs/court_pose/` a chaque epoque
  (`weights/best.pt` = le meilleur modele).
- Entrainement plafonne a **8 h**. **Si Colab coupe** : remets le GPU puis *Tout executer* -> reprise auto.

In [ ]:
# 1. GPU ?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive (dossier CourtVision_court) + Ultralytics
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CourtVision_court'
os.makedirs(DRIVE, exist_ok=True)
print('Dossier Drive :', DRIVE, '->', os.listdir(DRIVE))
!pip -q install -U ultralytics

In [ ]:
# 3. Dataset : copie du zip depuis le Drive puis decompression (~2-5 min)
import os, shutil, zipfile
DATA = '/content/court_pose'
if not os.path.exists(f'{DATA}/data.yaml'):
    src = f'{DRIVE}/court_pose.zip'
    if not os.path.exists(src):
        raise SystemExit("court_pose.zip introuvable dans Mon Drive/CourtVision_court : depose-le d'abord.")
    shutil.copy(src, '/content/court_pose.zip')
    zipfile.ZipFile('/content/court_pose.zip').extractall('/content')
    os.remove('/content/court_pose.zip')
    with open(f'{DATA}/data.yaml') as f:
        y = f.read()
    with open(f'{DATA}/data.yaml', 'w') as f:
        f.write(f'path: {DATA}\n' + y)
for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{DATA}/images/{s}')), 'images')

In [ ]:
# 4. Entrainement YOLO11s-pose -- reprend automatiquement si une session precedente a ete coupee
import os
from ultralytics import YOLO

RUNS = f'{DRIVE}/runs'
last = f'{RUNS}/court_pose/weights/last.pt'
if os.path.exists(last):
    print('Reprise de', last)
    YOLO(last).train(resume=True)
else:
    YOLO('yolo11s-pose.pt').train(
        data=f'{DATA}/data.yaml',
        imgsz=960,           # le terrain est un gros objet : 960 suffit, et l'inference reste rapide
        epochs=150,          # plafond theorique ; 'time' coupe avant
        time=8,              # heures max
        batch=16,
        workers=os.cpu_count(),
        cache=False,
        project=RUNS,
        name='court_pose',
        patience=25,
        # Augmentations adaptees a une camera TV : pas de mosaique (elle decoupe le terrain),
        # pas de rotation ; miroir gauche/droite OK (flip_idx dans data.yaml)
        mosaic=0.0, degrees=0.0, scale=0.3, translate=0.1, fliplr=0.5, flipud=0.0,
        plots=True,
    )

In [ ]:
# 5. Evaluation sur val et test (videos jamais vues) + erreur moyenne en pixels par point
import json, glob, numpy as np
best = YOLO(f'{RUNS}/court_pose/weights/best.pt')
out = {}
for split in ('val', 'test'):
    m = best.val(data=f'{DATA}/data.yaml', split=split, imgsz=960, batch=16,
                 project=RUNS, name=f'court_pose_eval_{split}', exist_ok=True, plots=False)
    out[split] = dict(pose_mAP50=float(m.pose.map50), pose_mAP50_95=float(m.pose.map),
                      box_mAP50=float(m.box.map50))

errs = []
for img in sorted(glob.glob(f'{DATA}/images/test/*.jpg')):
    lab = np.array(open(img.replace('/images/', '/labels/').replace('.jpg', '.txt')).read().split()[5:], float).reshape(14, 3)
    r = best.predict(img, imgsz=960, verbose=False)[0]
    if r.keypoints is None or len(r.boxes) == 0:
        continue
    pred = r.keypoints.xy[int(r.boxes.conf.argmax())].cpu().numpy()
    h, w = r.orig_shape
    vis = lab[:, 2] > 0
    errs += list(np.linalg.norm(pred[vis] - lab[vis, :2] * [w, h], axis=1))
errs = np.array(errs)
out['test_erreur_px'] = dict(mediane=round(float(np.median(errs)), 2), p90=round(float(np.quantile(errs, 0.9)), 2))
print(json.dumps(out, indent=2))
open(f'{RUNS}/court_pose/final_metrics.json', 'w').write(json.dumps(out, indent=2))
print(f'\nPoids a recuperer : Mon Drive/CourtVision_court/runs/court_pose/weights/best.pt')